# Member 3 - Task 1: Data audit (Kaggle notebook)

**What this notebook does:** it checks the *data*, not the model. It never trains anything and never edits the
team's shared files. It answers four questions:

1. Are train / validation / test really separate, and are labelled / unlabelled images what we think they are?
2. Why does one Galaxy10 file have 17,736 images and another 17,675?
3. Is the extra file `decals_images.h5` safe to use? (Default answer: **no**, until the evidence says otherwise.)
4. Can we save the exact image indices, labels, seeds and hashes so everyone can reproduce the split?

**How to use it:** run the cells from top to bottom. **Settings -> Internet must be ON** (for `git clone`).
No GPU is needed. Step 4 is the only one you must edit (file paths).

## Step 1 - Get the team's code (read-only)
*In simple words:* we download the repository so this notebook can reuse Member 2's functions (`split_indices`,
`check_cutout_overlap`, ...) instead of copying them. Before running this, push `data_audit.py` and
`test_data_audit.py` to your branch (`data-evaluation`) and change `BRANCH` below if you used another name.

In [ ]:
import os, sys, subprocess
REPO   = "/kaggle/working/repo"
BRANCH = "data-evaluation"      # <- the branch where you pushed data_audit.py + test_data_audit.py
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH,
                    "https://github.com/purveee04/Research_Methodology", REPO], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)
print([f for f in sorted(os.listdir(".")) if f.endswith(".py")])
try:
    import training_module as tm
except ImportError as e:                      # Kaggle TF >= 2.16 needs the legacy Keras package
    print(e); subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf_keras"], check=True)
    import training_module as tm
import data_audit as da
import numpy as np, h5py, json

## Step 2 - Run the unit tests first
*In simple words:* the tests use tiny fake data and prove the audit code itself behaves (for example: it *does*
notice a validation image hidden in the training pool). If anything here says `FAILED`, stop and tell me.
You should see `Ran 29 tests ... OK` and `Ran 36 tests ... OK`.

In [ ]:
for mod in (["test_data_audit"],
            ["test_training_module.TestSplitAndPositions", "test_training_module.TestCutoutOverlap",
             "test_training_module.TestFingerprints", "test_training_module.TestResumeManifest"]):
    r = subprocess.run([sys.executable, "-m", "unittest", *mod], capture_output=True, text=True)
    print(" ".join(mod)); print(r.stderr[-700:]); print("-" * 60)

## Step 3 - Find every HDF5 file Kaggle gave this notebook
*In simple words:* the same file name can hide **different files** (the baseline notebook saw a 20,000-image
`decals_images.h5`; Member 2 saw a 1,178-image one). This lists every `.h5` with its size and image shape so you
can pick the right paths in Step 4. Attach all candidate datasets with **Add Input** before running.

In [ ]:
rows = []
for root, _, files in os.walk("/kaggle/input"):
    for fn in files:
        if fn.endswith((".h5", ".hdf5")):
            p = os.path.join(root, fn)
            with h5py.File(p, "r") as f:
                rows.append((p, round(os.path.getsize(p) / 1e9, 3),
                             tuple(f["images"].shape) if "images" in f else None, sorted(f.keys())))
for r in sorted(rows):
    print(r)

## Step 4 - EDIT THIS CELL: say which file is which
*In simple words:* tell the notebook which file Member 2 trained with, which one the baseline used (if you can
attach it), and which extra-cutout files exist. Leave `None` / delete lines you do not have. Use the shapes printed
in Step 3: Galaxy10 should be `(N, 256, 256, 3)`; write down N for each.

In [ ]:
G10_CURRENT  = "/kaggle/input/REPLACE_ME/Galaxy10_DECaLS.h5"   # file Member 2's smoke run used (17,675 images?)
G10_BASELINE = None   # file the baseline notebook used (17,736 images?). e.g. "/kaggle/input/.../Galaxy10_DECaLS.h5"
EXTRA_FILES  = {      # every decals_images.h5 you have: name -> path
    # "extra_20000_256px": "/kaggle/input/.../decals_images.h5",
    # "extra_1178_69px":   "/kaggle/input/.../decals_images.h5",
}
OUT = "/kaggle/working/member3_audit"
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(G10_CURRENT), "fix G10_CURRENT (use a path printed in Step 3)"

# Same data settings as Member 2's trainer defaults. Do NOT change them here unless the team changes them.
DATA_SETTINGS = tm.make_data_settings(split_seed=16, test_frac=0.1, val_frac=0.1, label_frac=0.125,
                                      include_labelled_in_unlabeled=False, no_pool_unlabeled=False)
print(DATA_SETTINGS)

## Step 5 - Audit the Galaxy10 file Member 2 used
*In simple words:* this (a) reads the file and records its SHA-256 fingerprint, (b) rebuilds the train / validation /
test / labelled / unlabelled split with Member 2's own `split_indices`, (c) proves the groups do not overlap,
(d) hashes every image to find copies, and (e) uses each galaxy's sky position to see whether a **training** galaxy
sits on top of a **validation/test** galaxy. It takes a few minutes (it reads about 3 GB).

How to read the output: `MEMBERSHIP OK` is what we need. Any `DUPLICATES ... across splits` or `SKY ... same centre`
number above 0 is a leakage risk to discuss with the team (do not ignore it).

In [ ]:
report, state = da.audit_galaxy10_file(G10_CURRENT, DATA_SETTINGS)
print("\n".join(da.summarize_galaxy10_report(report)))
with open(f"{OUT}/galaxy10_current_audit.json", "w") as f:
    f.write(da.to_json(report))

## Step 6 - The 17,736 vs 17,675 question
*In simple words:* if you attached both Galaxy10 files, this finds exactly which images differ: removed images,
images whose pixels changed, copies of the same picture, and whether any **labels** changed. It matters because the
split uses row numbers: if 61 rows are missing from the middle of the file, the same seed picks **different
images**, so results from the two files are not comparable.

In [ ]:
cmp_report = None
if G10_BASELINE:
    rep_b, st_b = da.audit_galaxy10_file(G10_BASELINE, DATA_SETTINGS)
    print("BASELINE FILE"); print("\n".join(da.summarize_galaxy10_report(rep_b)))
    radec = lambda s: (s["ra"], s["dec"]) if s["ra"] is not None and s["dec"] is not None else None
    cmp_report = da.compare_galaxy10_versions(st_b["hashes"], state["hashes"], st_b["labels"], state["labels"],
                                              radec(st_b), radec(state))
    print("\nBASELINE (A) vs CURRENT (B)")
    print(da.to_json({k: v for k, v in cmp_report.items() if not k.endswith("_indices") and not k.endswith("_in_a")}))
    print("removed rows in A:", cmp_report.get("removed_indices_in_a", cmp_report["unmatched_a_indices"])[:80])
    with open(f"{OUT}/galaxy10_version_comparison.json", "w") as f:
        f.write(da.to_json({"baseline": rep_b, "comparison": cmp_report}))
else:
    print("G10_BASELINE is None: attach the 17,736-image file and set it in Step 4 to answer this question.")

## Step 7 - Is `decals_images.h5` safe? (go / no-go per file)
*In simple words:* for every extra file we check, in order: right image size for the loader's crop? sky coordinates
stored? pixel scale stored **in the file** (a number typed on the command line is not accepted here)? does any
cutout's footprint touch a validation/test galaxy? is any cutout byte-identical to a validation/test image?
**One failure = `NOT_APPROVED`.** Even `CLEARED_PENDING_SIGNOFF` only means "no overlap found by the metadata
check" - the team still has to agree.

In [ ]:
external = {}
for name, path in EXTRA_FILES.items():
    print("=" * 70); print(name, path)
    eh = da.scan_images(path)["hashes"]
    external[name] = da.check_external_cutouts(path, G10_CURRENT, state["ix"], extra_hashes=eh,
                                               g10_hashes=state["hashes"])
    e = external[name]
    print("shape: %d images of %dx%d | loader crop would keep %s px | has ra/dec: %s | has pxscale: %s" % (
        e["file"]["n_images"], e["file"]["height"], e["file"]["width"], e["file"]["loader_crop_hw"],
        e["file"]["has_ra_dec"], e["file"]["has_pxscale"]))
    if e["overlap"]:
        print("overlap check [%s]: %s" % (e["overlap"]["mode"], e["overlap"]["message"]))
    print("exact duplicates of val/test images:", e["n_exact_duplicates_of_val_test"])
    print("VERDICT:", e["verdict"]["status"])
    for why in e["verdict"]["reasons"]:
        print("   -", why)
with open(f"{OUT}/external_cutout_verdicts.json", "w") as f:
    f.write(da.to_json(external))
if not EXTRA_FILES:
    print("No extra files listed in Step 4 - nothing to check.")

## Step 8 - Save the split so everyone gets the same images
*In simple words:* writes three files: `split_indices.npz` (the exact index lists + labels), `split_table.csv`
(one row per image: index, label, train/val/test, which stream it is in, content hash, sky position) and
`split_manifest.json` (seeds, fractions, hashes, library versions). Then it **re-creates** the split from scratch and
checks it matches. If `reproduced` is `False`, the split changed (different labels, settings, or scikit-learn
version) and the saved index lists, not the seed, must be treated as the truth.

In [ ]:
identity = tm.dataset_identity(G10_CURRENT, None, state["labels"], state["ix"], state["unl_idx"])
SPLIT_DIR = f"{OUT}/split_seed{DATA_SETTINGS['split_seed']}"
paths = da.write_split_artifacts(SPLIT_DIR, state["labels"], state["ix"], state["unl_idx"], DATA_SETTINGS,
                                 identity=identity, content_hashes=state["hashes"],
                                 ra=state["ra"], dec=state["dec"], source_file=G10_CURRENT)
print(paths)
print(da.verify_split_artifacts(SPLIT_DIR, state["labels"], DATA_SETTINGS))

## Step 9 - Package the results to send to the team
*In simple words:* one zip with every JSON report and the split files. Download it from the **Output** tab, put it
in the GitHub branch (or a shared Kaggle dataset), and paste the short summary printed below into the team chat.

In [ ]:
import shutil
zip_path = shutil.make_archive("/kaggle/working/member3_audit_results", "zip", OUT)
print("ZIP:", zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")
print("\n=== SUMMARY TO PASTE ===")
print("Galaxy10 file:", G10_CURRENT)
print("\n".join(da.summarize_galaxy10_report(report)))
print("split reproduced:", da.verify_split_artifacts(SPLIT_DIR, state["labels"], DATA_SETTINGS)["reproduced"])
if cmp_report:
    print("vs baseline file: %d vs %d images | unmatched in A: %d | unmatched in B: %d | label changes: %s" % (
        cmp_report["n_a"], cmp_report["n_b"], cmp_report["n_unmatched_a"], cmp_report["n_unmatched_b"],
        cmp_report.get("n_matched_with_changed_label")))
for k, v in external.items():
    print("extra file %s: %s" % (k, v["verdict"]["status"]))